In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time
import os

# Guarantee absolute reproducibility across runs
os.environ['TF_DETERMINISTIC_OPS'] = '1'
tf.random.set_seed(42)
np.random.seed(42)

print("All  foundational model factory dependencies successfully loaded.")

In [ ]:
# [CELL 2] GLOBAL EXPERIMENT CONFIGURATION
CONFIG = {
    # Dataset Configuration
    "FILE_PATH": "BTC OHLCV Data.csv",  # Path to the cryptocurrency OHLCV dataset

    # Model Selection Architecture
    # Options: 'LSTM', 'GRU', 'TCN','DEEPAR', 'N-BEATS', 'N-HITS' 'TFT' 'INFORMER' 'AUTOFORMER''DLINEAR'
    #'XLSTM''TRANSFORMER''PATCHTST''iTRANSFORMER''BI-LSTM''CNN''TIMEMIXER''TIDE''TIMEXER''KOOPA'
    "MODEL_TYPE": "XLSTM",

    # Time-Series Windowing Parameters
    "LOOK_BACK": 60,                        # Input window:)
    "TARGET_STEPS": 2,                     # Multi-step Horizon: Next 2 hours
    # Data Splitting & Training Hyperparameters
    "TRAIN_SPLIT": 0.80,                    # 80% Training, 20% Testing (Chronological split)
    "BATCH_SIZE": 128,                      # Number of samples per gradient update
    "EPOCHS": 50,                           # Maximum number of training passes

    # Feature Engineering Parameters
    "FEATURES": ["Open", "High", "Low", "Close", "Volume"],
    "TARGET_FEATURE": "Close",              # The specific metric to forecast

    # Thesis Artifacts Directory
    "OUTPUT_DIR": "./thesis_results"
}

# Ensure directory exists for saving publication graphs & CSV metrics
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

# Apply a clean, standard styling context for high-resolution publication-ready figures
plt.style.use('default')

print(f" Cell 2 executed successfully: Configuration initialized for {CONFIG['MODEL_TYPE']} model.")

In [ ]:
# [CELL 3] EXPLORATORY DATA ANALYSIS (EDA) ENGINE
def run_publication_eda(df, config):
    """
    Generates high-resolution, publication-ready statistical visualizations
    and saves core metrics into the designated output directory.
    """
    print("## Running Dataset Structural Diagnostics...")
    print(f"Shape of Dataset: {df.shape}")
    print("\nData Types & Info:")
    df.info()

    # 1. Statistical Summary (Saved as CSV for Thesis Tables)
    summary = df[config["FEATURES"]].describe()
    summary_path = os.path.join(config["OUTPUT_DIR"], "statistical_summary.csv")
    summary.to_csv(summary_path)
    print(f"\n Statistical summary saved to: {summary_path}")

    # 2. Correlation Matrix Heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(df[config["FEATURES"]].corr(), annot=True, cmap="coolwarm", fmt=".4f", square=True)
    plt.title("Feature Correlation Matrix", fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(config["OUTPUT_DIR"], "eda_correlation_matrix.png"), dpi=300)
    plt.show()

    # 3. Macro Trends: Closing Price & Volume Over Time
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Target Feature (Close Price)
    axes[0].plot(df.index, df[config["TARGET_FEATURE"]], color='#1f77b4', linewidth=1)
    axes[0].set_title("Bitcoin Historical Closing Price Trend", fontweight='bold', fontsize=12)
    axes[0].set_ylabel("Price (USDT)")
    axes[0].grid(True, linestyle='--', alpha=0.5)

    # Trading Volume
    axes[1].fill_between(df.index, df["Volume"], color='#ff7f0e', alpha=0.5)
    axes[1].set_title("Historical Trading Volume", fontweight='bold', fontsize=12)
    axes[1].set_ylabel("Volume")
    axes[1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(config["OUTPUT_DIR"], "eda_macro_trends.png"), dpi=300)
    plt.show()

    # 4. Feature Distributions Histograms
    df[config["FEATURES"]].hist(bins=50, figsize=(14, 9), color='darkblue', grid=True, edgecolor='black', alpha=0.7)
    plt.suptitle("Feature Distributions Analysis", fontweight='bold', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(config["OUTPUT_DIR"], "eda_feature_distributions.png"), dpi=300)
    plt.show()

print("Cell 3 executed successfully: Publication-ready EDA engine defined.")

In [ ]:
# [CELL 4] DATA PREPARATION & MULTI-STEP SEQUENCE ENGINE
def load_and_index_dataset(config):
    """
    Loads the financial time-series data and establishes a clean
    DatetimeIndex to handle rigorous chronological ordering.
    """
    df = pd.read_csv(config["FILE_PATH"])
    # Automatically locate date or time column
    date_col = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
    if date_col:
        # Robust handling of mixed timestamps formats (with or without milliseconds)
        df[date_col[0]] = pd.to_datetime(df[date_col[0]], format='mixed')
        df.set_index(date_col[0], inplace=True)
    return df

def generate_sequences(data, look_back, target_idx, target_steps):
    """
    Generates input-output windows.
    CRITICAL FIX FOR MULTI-STEP FORECASTING:
    y now captures a continuous vector of 'target_steps' (24 intervals / 2 hours)
    instead of a single shifted scalar index.
    """
    X, y = [], []

    # Adjusted boundary to allow a continuous chunk of length 'target_steps' ahead
    for i in range(len(data) - look_back - target_steps + 1):
        # Input features window: Shape (look_back, num_features)
        X.append(data[i : (i + look_back), :])

        # Output target vector sequence: Next 24 steps continuous close prices
        # Shape: (target_steps,)
        y.append(data[(i + look_back) : (i + look_back + target_steps), target_idx])

    return np.array(X), np.array(y)

def prep_thesis_data(df, config):
    """
    Transforms raw dataframe into normalized training and testing tensors
    while strictly avoiding forward-looking/data leakage bias.
    """
    data_matrix = df[config["FEATURES"]].values
    target_idx = config["FEATURES"].index(config["TARGET_FEATURE"])

    # Chronological training/testing split (No random shuffling to preserve time dependency)
    split_boundary = int(len(data_matrix) * config["TRAIN_SPLIT"])
    train_data = data_matrix[:split_boundary]
    test_data = data_matrix[split_boundary:]

    # Fit scaler ONLY on training data to mimic a realistic deployment scenario
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_train = scaler.fit_transform(train_data)
    scaled_test = scaler.transform(test_data)

    # Build sequence pairs for training tensor
    X_train, y_train = generate_sequences(
        scaled_train,
        config["LOOK_BACK"],
        target_idx,
        config["TARGET_STEPS"]
    )

    # Build sequence pairs for evaluation tensor
    X_test, y_test = generate_sequences(
        scaled_test,
        config["LOOK_BACK"],
        target_idx,
        config["TARGET_STEPS"]
    )

    return X_train, y_train, X_test, y_test, scaler, target_idx

print("Cell 4 executed successfully: Multi-step data preparation pipeline fully optimized.")

In [ ]:
# [CELL 5] UNIFIED COMPREHENSIVE MODEL FACTORY (20 ARCHITECTURES)
def build_neural_architecture(model_type, input_shape, target_steps):
    """
    Academic Factory Pattern generating 20 state-of-the-art time-series models.
    Inputs: input_shape -> (LOOK_BACK, FEATURES), target_steps -> (24)
    """
    inputs = layers.Input(shape=input_shape)
    look_back = input_shape[0]
    num_features = input_shape[1]

    # Convert model_type to uppercase to prevent casing mismatches
    model_type = model_type.upper()

    
    # 1. STANDARD LSTM
   
    if model_type == "LSTM":
        x = layers.LSTM(64, return_sequences=True)(inputs)
        x = layers.Dropout(0.2)(x)
        x = layers.LSTM(32, return_sequences=False)(x)
        x = layers.Dropout(0.2)(x)
        outputs = layers.Dense(target_steps, name="LSTM_Output")(x)

    
    # 2. GRU (Gated Recurrent Unit)
    
    elif model_type == "GRU":
        x = layers.GRU(64, return_sequences=True)(inputs)
        x = layers.Dropout(0.2)(x)
        x = layers.GRU(32, return_sequences=False)(x)
        x = layers.Dropout(0.2)(x)
        outputs = layers.Dense(target_steps, name="GRU_Output")(x)

   
    # 3. TCN (Temporal Convolutional Network)
    
    elif model_type == "TCN":
        x = layers.Conv1D(filters=32, kernel_size=3, padding='causal', dilation_rate=1, activation='relu')(inputs)
        x = layers.LayerNormalization()(x)
        x = layers.Conv1D(filters=64, kernel_size=3, padding='causal', dilation_rate=2, activation='relu')(x)
        x = layers.LayerNormalization()(x)
        x = layers.Conv1D(filters=128, kernel_size=3, padding='causal', dilation_rate=4, activation='relu')(x)
        x = layers.GlobalAveragePooling1D()(x)
        x = layers.Dropout(0.2)(x)
        outputs = layers.Dense(target_steps, name="TCN_Output")(x)

    
    # 4. DEEPAR (Deep Autoregressive Recurrent Network)
    
    elif model_type == "DEEPAR":
        x = layers.LSTM(128, return_sequences=True)(inputs)
        x = layers.LayerNormalization()(x)
        x = layers.LSTM(64, return_sequences=False)(x)
        x = layers.Dense(64, activation='relu')(x)
        outputs = layers.Dense(target_steps, name="DeepAR_Deterministic_Output")(x)

    
    # 5. N-BEATS (Neural Basis Expansion Analysis)
    
    elif model_type == "N-BEATS":
        flat_in = layers.Flatten()(inputs)
        x = layers.Dense(256, activation="relu")(flat_in)
        x = layers.Dense(256, activation="relu")(x)
        backcast = layers.Dense(look_back * num_features)(x)
        resid = layers.Subtract()([flat_in, backcast])
        x_fore = layers.Dense(128, activation="relu")(resid)
        outputs = layers.Dense(target_steps, name="N-BEATS_Output")(x_fore)

    
    # 6. N-HITS (Neural Hierarchical Interpolation)
    
    elif model_type == "N-HITS":
        pool_1 = layers.AveragePooling1D(pool_size=2, padding='same')(inputs)
        flat_1 = layers.Flatten()(pool_1)
        x = layers.Dense(128, activation="relu")(flat_1)
        outputs = layers.Dense(target_steps, name="N-HITS_Output")(x)

    
    # 7. TFT (Temporal Fusion Transformer)
    
    elif model_type == "TFT":
        attn = layers.MultiHeadAttention(num_heads=4, key_dim=num_features, dropout=0.1)(inputs, inputs)
        x = layers.Add()([inputs, attn])
        x = layers.LayerNormalization()(x)
        x = layers.GlobalAveragePooling1D()(x)
        x = layers.Dense(64, activation='relu')(x)
        outputs = layers.Dense(target_steps, name="TFT_Output")(x)

    
    # 8. INFORMER
    
    elif model_type == "INFORMER":
        attn = layers.MultiHeadAttention(num_heads=2, key_dim=num_features)(inputs, inputs)
        distill = layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')(attn)
        distill = layers.MaxPooling1D(pool_size=2)(distill)
        x = layers.Flatten()(distill)
        outputs = layers.Dense(target_steps, name="Informer_Output")(x)

    
    # 9. AUTOFORMER
    
    elif model_type == "AUTOFORMER":
        trend_part = layers.AveragePooling1D(pool_size=3, strides=1, padding='same')(inputs)
        seasonal_part = layers.Subtract()([inputs, trend_part])
        flat_s = layers.Flatten()(seasonal_part)
        x_s = layers.Dense(64, activation='relu')(flat_s)
        outputs = layers.Dense(target_steps, name="Autoformer_Output")(x_s)


    # 10. DLINEAR (Decomposition Linear Model)
    
    elif model_type == "DLINEAR":
        trend_comp = layers.AveragePooling1D(pool_size=5, strides=1, padding='same')(inputs)
        seasonal_comp = layers.Subtract()([inputs, trend_comp])
        flat_trend = layers.Flatten()(trend_comp)
        flat_seas = layers.Flatten()(seasonal_comp)
        out_trend = layers.Dense(target_steps)(flat_trend)
        out_seas = layers.Dense(target_steps)(flat_seas)
        outputs = layers.Add(name="DLinear_Output")([out_trend, out_seas])

    
    # 11. xLSTM (Extended Long Short-Term Memory)
    
    elif model_type == "XLSTM":
        norm_in = layers.LayerNormalization()(inputs)
        lstm_out = layers.LSTM(64, return_sequences=True)(norm_in)
        gate = layers.Dense(64, activation='sigmoid')(norm_in)
        gated_context = layers.Multiply()([lstm_out, gate])
        x = layers.GlobalAveragePooling1D()(gated_context)
        outputs = layers.Dense(target_steps, name="xLSTM_Output")(x)


    # NEW ADDITIONS (MODELS 12 - 20)


    # 12. STANDARD TRANSFORMER

    elif model_type == "TRANSFORMER":
        # Classic Vanilla Transformer encoder structure
        attn = layers.MultiHeadAttention(num_heads=4, key_dim=num_features)(inputs, inputs)
        x = layers.Add()([inputs, attn])
        x = layers.LayerNormalization()(x)
        # Feed-forward network block
        ffn = layers.Dense(num_features, activation="relu")(x)
        x = layers.Add()([x, ffn])
        x = layers.LayerNormalization()(x)
        x = layers.Flatten()(x)
        outputs = layers.Dense(target_steps, name="Transformer_Output")(x)


    # 13. PATCHTST (Patch Time Series Transformer)

    elif model_type == "PATCHTST":
        # Sub-sampling lookback window into local patches (simulated via 1D Strided Conv)
        patch_proj = layers.Conv1D(filters=32, kernel_size=4, strides=4, padding='same', activation='relu')(inputs)
        attn = layers.MultiHeadAttention(num_heads=2, key_dim=32)(patch_proj, patch_proj)
        x = layers.Add()([patch_proj, attn])
        x = layers.LayerNormalization()(x)
        x = layers.Flatten()(x)
        outputs = layers.Dense(target_steps, name="PatchTST_Output")(x)


    # 14. ITRANSFORMER (Inverted Transformer)

    elif model_type == "ITRANSFORMER":
        # Inverts tokens to represent variates rather than temporal steps
        inverted_inputs = layers.Permute((2, 1))(inputs) # Shape becomes (FEATURES, LOOK_BACK)
        attn = layers.MultiHeadAttention(num_heads=2, key_dim=look_back)(inverted_inputs, inverted_inputs)
        x = layers.Add()([inverted_inputs, attn])
        x = layers.LayerNormalization()(x)
        x = layers.Flatten()(x)
        outputs = layers.Dense(target_steps, name="iTransformer_Output")(x)

    # 15. BI-LSTM (Bidirectional LSTM)
    
    elif model_type == "BI-LSTM":
        x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(inputs)
        x = layers.Dropout(0.2)(x)
        x = layers.Bidirectional(layers.LSTM(32, return_sequences=False))(x)
        x = layers.Dropout(0.2)(x)
        outputs = layers.Dense(target_steps, name="Bi-LSTM_Output")(x)


    # 16. CNN (Convolutional Neural Network Standard Baseline)

    elif model_type == "CNN":
        x = layers.Conv1D(filters=64, kernel_size=3, padding='same', activation='relu')(inputs)
        x = layers.MaxPooling1D(pool_size=2)(x)
        x = layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')(x)
        x = layers.GlobalAveragePooling1D()(x)
        outputs = layers.Dense(target_steps, name="CNN_Output")(x)


    # 17. TIMEMIXER

    elif model_type == "TIMEMIXER":
        # Multiscale mixing structures implemented through multi-resolution downsamplings
        scale_1 = layers.AveragePooling1D(pool_size=2, padding='same')(inputs)
        scale_2 = layers.AveragePooling1D(pool_size=4, padding='same')(inputs)
        # Fully mixed representations using local project weights
        flat_in = layers.Flatten()(inputs)
        flat_s1 = layers.Flatten()(scale_1)
        flat_s2 = layers.Flatten()(scale_2)
        mixed = layers.Concatenate()([flat_in, flat_s1, flat_s2])
        x = layers.Dense(128, activation='relu')(mixed)
        outputs = layers.Dense(target_steps, name="TimeMixer_Output")(x)


    # 18. TIDE (Time-series Dense Encoder)

    elif model_type == "TIDE":
        # Extremely fast multi-layer perceptron dense encoder-decoder architecture
        flat_in = layers.Flatten()(inputs)
        encoder = layers.Dense(256, activation='relu')(flat_in)
        encoder = layers.Dense(128, activation='relu')(encoder)
        decoder = layers.Dense(64, activation='relu')(encoder)
        outputs = layers.Dense(target_steps, name="TiDE_Output")(decoder)


    # 19. TIMEXER

    elif model_type == "TIMEXER":
        # Explicitly captures cross-variate dependencies through spatial embedding maps
        spatial_emb = layers.Dense(num_features, activation='relu')(inputs)
        attn = layers.MultiHeadAttention(num_heads=2, key_dim=num_features)(spatial_emb, spatial_emb)
        x = layers.Add()([spatial_emb, attn])
        x = layers.Flatten()(x)
        outputs = layers.Dense(target_steps, name="TimeXer_Output")(x)


    # 20. KOOPA

    elif model_type == "KOOPA":
        # Learning non-stationary dynamics via Koopman Operator approximations
        flat_in = layers.Flatten()(inputs)
        # Linear operator layer simulating operator transitions
        koopman_operator = layers.Dense(128, use_bias=False)(flat_in)
        x = layers.Dense(64, activation='relu')(koopman_operator)
        outputs = layers.Dense(target_steps, name="Koopa_Output")(x)

    else:
        raise ValueError(f" Model Type '{model_type}' is missing from the Cell 5 Factory configuration.")

    # Generate complete architectural model container
    model = models.Model(inputs=inputs, outputs=outputs, name=f"{model_type}_Execution_Engine")

    # Establish academic compilation standards using optimized learning rates
    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"]
    )
    return model

print("Master Factory updated. All 20 deep learning pipelines are fully active and uniform!")

In [ ]:
# [CELL 6] VALIDATION & EVALUATION METRICS ENGINE
def execute_walk_forward_validation(df, config, folds=3):
    """
    Executes Walk-Forward Time-Series Cross-Validation to evaluate model
    generalization over continuous temporal chunks without data leakage.
    """
    print("## Running Walk-Forward Time-Series Cross Validation...")
    data_matrix = df[config["FEATURES"]].values
    target_idx = config["FEATURES"].index(config["TARGET_FEATURE"])
    fold_size = len(data_matrix) // (folds + 1)
    fold_scores = []

    for f in range(folds):
        train_end = fold_size * (f + 1)
        test_end = train_end + fold_size

        train_block = data_matrix[:train_end]
        test_block = data_matrix[train_end:test_end]

        fold_scaler = MinMaxScaler()
        scaled_train = fold_scaler.fit_transform(train_block)
        scaled_test = fold_scaler.transform(test_block)

        # Generate multi-step structures using the updated sequence rules
        X_tr, y_tr = generate_sequences(scaled_train, config["LOOK_BACK"], target_idx, config["TARGET_STEPS"])
        X_te, y_te = generate_sequences(scaled_test, config["LOOK_BACK"], target_idx, config["TARGET_STEPS"])

        # CRITICAL FIX: Pass target_steps explicitly to the model architecture factory
        fold_model = build_neural_architecture(
            config["MODEL_TYPE"],
            (config["LOOK_BACK"], len(config["FEATURES"])),
            target_steps=config["TARGET_STEPS"]
        )

        # Fast training on validation folds to save memory and time
        fold_model.fit(X_tr, y_tr, epochs=3, batch_size=config["BATCH_SIZE"], verbose=0)

        preds = fold_model.predict(X_te, verbose=0)
        fold_mse = mean_squared_error(y_te, preds)
        fold_scores.append(fold_mse)
        print(f"Fold {f+1}/{folds} Normalized Multi-step MSE: {fold_mse:.6f}")

    return np.mean(fold_scores)

def calculate_statistical_metrics(y_true, y_pred):
    """
    Computes all 9 rigorous thesis accuracy matrices for multi-step predictions.
    Flattens vectors to ensure absolute dimensional safety across continuous series.
    """
    # Flatten the 2D multi-step arrays to standard 1D continuous trajectories
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()

    # Core Regression Errors
    mse = mean_squared_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true_flat, y_pred_flat)

    # Percentage Errors
    mape = np.mean(np.abs((y_true_flat - y_pred_flat) / (y_true_flat + 1e-8))) * 100
    smape = 100 * np.mean(2 * np.abs(y_pred_flat - y_true_flat) / (np.abs(y_true_flat) + np.abs(y_pred_flat) + 1e-8))

    # Goodness of fit
    r2 = r2_score(y_true_flat, y_pred_flat)

    # Restored Advanced Thesis Metrics
    max_err = np.max(np.abs(y_true_flat - y_pred_flat))
    median_ae = np.median(np.abs(y_true_flat - y_pred_flat))

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "Median_AE": median_ae,
        "Max_Error": max_err,
        "MAPE(%)": mape,
        "SMAPE(%)": smape,
        "R2": r2,
    }

print(" Cell 6 executed successfully: All 9 multi-step evaluation metrics fully integrated.")

In [ ]:
# [CELL 7] LOAD DATA & RUN EDA
try:
    raw_df = load_and_index_dataset(CONFIG)
    print(" Dataset loaded successfully from specified path.")
except FileNotFoundError:
    print(f"Dataset not found at {CONFIG['FILE_PATH']}. Generating synthetic dummy data for pipeline testing.")
    # Generating 5-minute interval timestamp index for continuity
    dummy_dates = pd.date_range(start="2021-01-01", periods=5000, freq="5min")
    raw_df = pd.DataFrame(np.random.randn(5000, 5), columns=CONFIG["FEATURES"], index=dummy_dates)
    # Creating a realistic geometric random walk for price simulation
    raw_df["Close"] = 50000 + raw_df["Close"].cumsum() * 10

# Trigger the academic EDA diagnostics and visualization suite
run_publication_eda(raw_df, CONFIG)
print(" Cell 7 executed successfully: Data pipeline ingestion and EDA diagnostics completed.")

In [ ]:
# [CELL 8] CROSS-VALIDATION EXECUTION & MAIN TENSOR TRAIN-TEST SPLIT
print("==============================================================")
print("     PHASE 1: EXECUTING TIME-SERIES CROSS-VALIDATION          ")
print("==============================================================")

# Execute walk-forward cross-validation to rigorously test for data leakage or decay
mean_cv_loss = execute_walk_forward_validation(raw_df, CONFIG, folds=3)
print(f"\n Overall Mean Cross-Validation MSE: {mean_cv_loss:.6f}")

print("\n==============================================================")
print("     PHASE 2: PREPARING PRODUCTION TRAIN-TEST TENSORS        ")
print("==============================================================")

# Extract and sequence main arrays for final thesis benchmark training
X_train, y_train, X_test, y_test, scaler, target_idx = prep_thesis_data(raw_df, CONFIG)

# Print explicit shape diagnostics to guarantee 24-step multi-step vector layout
print("\n Verification of Multi-Step Tensor Structural Dimensions:")
print(f"• X_train shape (Samples, Lookback, Features): {X_train.shape}")
print(f"• y_train shape (Samples, Target Steps):       {y_train.shape} -> [Targeting next 24 intervals/2 hours]")
print(f"• X_test shape  (Samples, Lookback, Features): {X_test.shape}")
print(f"• y_test shape  (Samples, Target Steps):       {y_test.shape} -> [Targeting next 24 intervals/2 hours]")

print("\n Cell 8 executed successfully: Dataset split and dimensions validated for multi-step forecasting.")

In [ ]:
# [CELL 9] CENTRALIZED MODEL TRAINING ENGINE & LOSS DIAGNOSTICS
# Check if CONFIG is defined. If not, raise an informative error.
if 'CONFIG' not in globals():
    raise NameError("The 'CONFIG' variable is not defined. Please ensure Cell 2 ('GLOBAL EXPERIMENT CONFIGURATION') has been executed successfully before running this cell.")

# Check if build_neural_architecture is defined. If not, raise an informative error.
if 'build_neural_architecture' not in globals():
    raise NameError("The 'build_neural_architecture' function is not defined. Please ensure Cell 5 ('UNIFIED COMPREHENSIVE MODEL FACTORY') has been executed successfully before running this cell.")

print("==============================================================")
print(f"     PHASE 3: TRAINING {CONFIG['MODEL_TYPE']} ARCHITECTURE")
print("==============================================================")

# 1. Instantiate the dynamically chosen model architecture from the Cell 5 Factory
input_shape = (CONFIG["LOOK_BACK"], len(CONFIG["FEATURES"]))
print(f" Building {CONFIG['MODEL_TYPE']} network structure...")
model = build_neural_architecture(CONFIG["MODEL_TYPE"], input_shape, CONFIG["TARGET_STEPS"])

# Print architectural topology layout for thesis verification
model.summary()

# 2. Set up academic callbacks (Early Stopping to safeguard against overfitting)
callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

# 3. Track precise execution duration and execute training loop
print(f"\n Initiating training loop for up to {CONFIG['EPOCHS']} epochs...")
start_train_time = time.time()

# Fit the model and store metrics progression inside the 'history' container
history = model.fit(
    X_train, y_train,
    epochs=CONFIG["EPOCHS"],
    batch_size=CONFIG["BATCH_SIZE"],
    validation_split=0.15,
    callbacks=callbacks_list,
    verbose=1
)

# Calculate training duration and store globally for Cell 10's efficiency report
training_duration = time.time() - start_train_time
print(f"\n Training cycle finalized in: {training_duration:.2f} seconds.")

print("\n==============================================================")
print("     PHASE 3.5: GENERATING LEARNING CURVE DIAGNOSTICS        ")
print("==============================================================")

# 4. Render High-Resolution Training vs Validation Loss Curves
plt.figure(figsize=(11, 5), dpi=300)

train_loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(train_loss) + 1)

# Plot curves with publication styling
plt.plot(epochs_range, train_loss, label="Training Loss (MSE)", color="#1f77b4", linewidth=2.2)
plt.plot(epochs_range, val_loss, label="Validation Loss (MSE)", color="#ff7f0e", linestyle="--", linewidth=2.2)

# Highlight final numerical terminal values on the plot canvas
final_train = train_loss[-1]
final_val = val_loss[-1]
plt.scatter(len(train_loss), final_train, color="#1f77b4", s=40, zorder=5)
plt.scatter(len(val_loss), final_val, color="#ff7f0e", s=40, zorder=5)

# Axis labels and standard thesis graph styling
plt.title(f"Learning Curve for {CONFIG['MODEL_TYPE']} Model", fontweight='bold', fontsize=14, pad=15)
plt.xlabel("Epoch", fontsize=11, labelpad=8)
plt.ylabel("Loss (MSE)", fontsize=11, labelpad=8)
plt.legend(loc="upper right", fontsize=10, frameon=True)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()

# Save the plot as a high-resolution image
learning_curve_path = os.path.join(CONFIG["OUTPUT_DIR"], f"{CONFIG['MODEL_TYPE']}_learning_curve.png")
plt.savefig(learning_curve_path, dpi=300)
plt.show()

print("\n Cell 9 executed successfully: Model trained and learning curve generated.")

In [ ]:
# [CELL 10] FULL 13-ELEMENT EVALUATION & THESIS REPORTING ENGINE
print("==============================================================")
print("     PHASE 4: COMPLETE SYSTEM EVALUATION & PERFORMANCE        ")
print("==============================================================")

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import time

# 1. Track Inference Efficiency Latency
print(f" Generating multi-step predictions using trained {CONFIG['MODEL_TYPE']} model...")
start_inf_time = time.time()
scaled_preds = model.predict(X_test, batch_size=CONFIG["BATCH_SIZE"], verbose=1)
inference_duration = time.time() - start_inf_time

# 2. Compute Structural Complexity Metrics
total_model_params = model.count_params()
# Estimate size in MB assuming standard float32 weights (4 bytes per parameter)
estimated_size_mb = (total_model_params * 4) / (1024 * 1024)

# Retrieve training time from global scope (fallback to 0.0 if Cell 9 wasn't run)
recorded_train_time = globals().get('training_duration', 0.0)

# 3. Perform Mathematical De-scaling to obtain actual prices (USDT)
num_features = len(CONFIG["FEATURES"])
target_idx = CONFIG["FEATURES"].index(CONFIG["TARGET_FEATURE"]) if CONFIG["TARGET_FEATURE"] in CONFIG["FEATURES"] else 0

print("\n Reversing min-max normalization to obtain actual Bitcoin prices (USDT)...")
def inverse_transform_multistep(scaled_matrix, scaler, target_idx, num_features):
    samples, steps = scaled_matrix.shape
    flat_scaled = scaled_matrix.reshape(-1, 1)
    dummy_matrix = np.zeros((flat_scaled.shape[0], num_features))
    dummy_matrix[:, target_idx] = flat_scaled[:, 0]
    inv_flat = scaler.inverse_transform(dummy_matrix)[:, target_idx]
    return inv_flat.reshape(samples, steps)

actual_prices = inverse_transform_multistep(y_test, scaler, target_idx, num_features)
predicted_prices = inverse_transform_multistep(scaled_preds, scaler, target_idx, num_features)

# Flatten arrays for point-by-point statistical validation metrics
actual_flat = actual_prices.flatten()
predicted_flat = predicted_prices.flatten()

# 4. Direct In-Cell Computation of the 9 Accuracy Metrics
print("\n Computing validation errors...")
mse = mean_squared_error(actual_flat, predicted_flat)
rmse = np.sqrt(mse)
mae = mean_absolute_error(actual_flat, predicted_flat)
median_ae = median_absolute_error(actual_flat, predicted_flat)
max_error = np.max(np.abs(actual_flat - predicted_flat))

# Safe calculation for percentage errors to prevent division-by-zero glitches
mape = np.mean(np.abs((actual_flat - predicted_flat) / np.where(actual_flat == 0, 1e-5, actual_flat))) * 100
smape = 200 * np.mean(np.abs(predicted_flat - actual_flat) / (np.abs(actual_flat) + np.abs(predicted_flat) + 1e-5))
r2 = r2_score(actual_flat, predicted_flat)

# Directional Accuracy computed per forecast step (not on overlapping flattened arrays)
last_known_scaled = X_test[:, -1, target_idx]  # last input timestep, scaled
dummy = np.zeros((len(last_known_scaled), num_features))
dummy[:, target_idx] = last_known_scaled
last_known_actual = scaler.inverse_transform(dummy)[:, target_idx]

step_das = []
for step in range(CONFIG["TARGET_STEPS"]):
    true_dir = np.sign(actual_prices[:, step] - last_known_actual)
    pred_dir = np.sign(predicted_prices[:, step] - last_known_actual)
    step_da = np.mean(true_dir == pred_dir) * 100
    step_das.append(step_da)
    print(f"   Step {step+1} Directional Accuracy: {step_da:.2f}%")

directional_accuracy = np.mean(step_das)

# 5. Compile All 13 Thesis Elements into a Single Master Dictionary
master_thesis_report = {
    # --- SECTION A: ACCURACY METRICS ---
    "MSE": mse,
    "RMSE": rmse,
    "MAE": mae,
    "Median_AE": median_ae,
    "Max_Error": max_error,
    "MAPE(%)": mape,
    "SMAPE(%)": smape,
    "R2_Score": r2,
    "Directional_Accuracy(%)": directional_accuracy,

    # --- SECTION B: COMPUTATIONAL EFFICIENCY METRICS ---
    "Training_Time(s)": recorded_train_time,
    "Inference_Time(s)": inference_duration,
    "Total_Model_Parameters": total_model_params,
    "Estimated_Model_Size(MB)": estimated_size_mb
}

# 6. Display Clean Summary Table
print("\n" + "="*55)
print(f"   MASTER THESIS BENCHMARK REPORT: {CONFIG['MODEL_TYPE']} ENGINE")
print("="*55)
print(f" {'METRIC ELEMENT':<30} | {'VALUE / SCORE':<20}")
print("-"*55)
for metric_key, value in master_thesis_report.items():
    if "%" in metric_key:
        print(f" • {metric_key:<28} | {value:.4f}%")
    elif "Time" in metric_key:
        print(f" • {metric_key:<28} | {value:.2f} seconds")
    elif "Parameters" in metric_key:
        print(f" • {metric_key:<28} | {int(value):,}")
    elif "Size" in metric_key:
        print(f" • {metric_key:<28} | {value:.4f} MB")
    else:
        print(f" • {metric_key:<28} | {value:.6f}")
print("======================================================")

# 7. Export Unified Results to CSV Spreadsheet
report_df = pd.DataFrame(list(master_thesis_report.items()), columns=["Thesis Metric Element", "Value Score"])
drive_folder = "./results"
os.makedirs(drive_folder, exist_ok=True)
csv_save_path = os.path.join(drive_folder, f"{CONFIG['MODEL_TYPE']}_complete_thesis_metrics.csv")
report_df.to_csv(csv_save_path, index=False)
print(f" Comprehensive metrics sheet successfully saved to: {csv_save_path}")

# 8. Plot Forecast Comparison Chart
plt.figure(figsize=(14, 6), dpi=300)
plot_tail = 150  # Focus window on the last 150 points for clear observation
plt.plot(actual_flat[-plot_tail:], label="Actual BTC Price (USDT)", color="#1f77b4", linewidth=2)
plt.plot(predicted_flat[-plot_tail:], label=f"Predicted BTC Price ({CONFIG['MODEL_TYPE']})",
         color="#d62728", linestyle="--", linewidth=1.8)

plt.title(f"Bitcoin Price Tracking Analysis ({CONFIG['MODEL_TYPE']} Multi-Step Framework)",
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Continuous Multi-Step Timeline Slices (1-Hour Steps)", fontsize=11, labelpad=8)
plt.ylabel("Price (USDT)", fontsize=11, labelpad=8)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper left", fontsize=10, frameon=True)
plt.tight_layout()

chart_save_path = os.path.join(CONFIG["OUTPUT_DIR"], f"{CONFIG['MODEL_TYPE']}_prediction_plot.png")
plt.savefig(chart_save_path, dpi=300)
plt.show()

print("\n Cell 10 executed successfully: All 13 metrics are captured, logged, and plotted.")